# Load Data

In [1]:
"""Load Data
Structure:
    1. Imports, Variables, Functions
    2. Load Data
"""

# 1. Imports, Variables, Functions
# imports   
import pandas as pd, numpy as np, os, sys
import anndata as ad
import logging
from typing import *
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.metrics import confusion_matrix, classification_report
import sys
sys.path.append(os.path.join("..", ".."))
from src.utils import utils as ut
from src.utils import viz as vz
from src.utils import io 
logging.basicConfig(level=logging.INFO)

# variables
# run_dir = os.path.join("..","..","outputs","run-25-09-28-05") 
run_dir = os.path.join("..","..","outputs","run-25-09-13-18") 
# run_dir = os.path.join("..","..","outputs","run-25-10-05-01") 
# run_dir = os.path.join("..","..","outputs","run-25-09-17-01") 

embedding_type = "ft"

assert embedding_type in ["ft", "pt", "raw"]
if embedding_type == "ft":
    output_dir = os.path.join(run_dir, "outputs")
elif embedding_type == "pt":
    run_name = run_dir.split("/")[-1]
    output_dir = os.path.join("/aloy/scratch/ddalton/projects/scGPT_playground/outputs/",run_name, "outputs")
elif embedding_type == "raw":
    run_name = run_dir.split("/")[-1]
    output_dir = os.path.join("/aloy/scratch/ddalton/projects/scGPT_playground/outputs/",run_name, "outputs")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# functions
def get_counts(df:pd.DataFrame, rel_map:Dict, key_interest:str)->pd.DataFrame:
    df_query = df.copy()

    n_same, n_rel, n_unrel, n_total, n_total_unique = [], [], [], [], []

    for _, r in df_query.iterrows():
        topk = list(r[key_interest])  # keep duplicates
        doid = r["query_doid"]
        related = rel_map.get(doid, set())

        same = sum(1 for x in topk if x == doid)           # absolute count of the exact disease
        rel  = sum(1 for x in topk if x in related)        # absolute count of related diseases (counts repeats)
        total = len(topk)
        unrel = total - same - rel

        n_same.append(same)
        n_rel.append(rel)
        n_unrel.append(unrel)
        n_total.append(total)
        n_total_unique.append(len(set(topk)))

    df_query["n_same"]  = n_same
    df_query["n_rel"]   = n_rel
    df_query["n_unrel"] = n_unrel
    df_query["n_total"] = n_total
    df_query["n_total_unique"] = n_total_unique
    
    df_query["pct_same"]  = df_query["n_same"] / df_query["n_total"] * 100
    df_query["pct_rel"]   = df_query["n_rel"] / df_query["n_total"] * 100
    df_query["pct_unrel"] = df_query["n_unrel"] / df_query["n_total"] * 100

    df_query["hits_same"] = (df_query["n_same"] > 0).astype(int)
    df_query["hits_rel"]  = (df_query["n_rel"] > 0).astype(int)

    # get precision
    df_query["prec@k"] = (df_query["n_rel"]+df_query["n_same"]) / df_query["n_total"]

    return df_query

# 2. Load Data
(
    # split,
    predictions_test,
    labels_test,
    results_test,
    all_outputs_test,
    predictions_valid,
    labels_valid,
    results_valid,
    all_outputs_valid,
    predictions_train,
    labels_train,
    results_train,
    all_outputs_train,
    adata_orig,
    id2type,
    train_indices,
    valid_indices,
) = io.load_run_output(run_dir)

# load json

with open(os.path.join(run_dir, "parameters.json"), "r") as f:
    parameters = json.load(f)

for k, v in parameters.items():
    print(f"{k}: {v}")

/home/ddalton/Data/old_home/miniconda3/envs/scgpt_2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Nº of loaded variables 16
data_path: /aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-09-12-01/data.h5ad
max_seq_len: 3501
batch_size: 16
gene_presence_pct: 0.9
benchmark_data: False
split_type: stratified
val_split_type: rand_stratified
n_splits: 10
n_tested_splits: 10
epochs: 50
gene_filtering: top_presence
sample_presence_pct: 0.3
MLM: False
CLS: False
CLS_multilabel: True
DAB: False
ADV: False
CCE: False
ecs_thres: 0.0
dab_weight: 0.0
use_fast_transformer: True
output_attentions: False
INPUT_BATCH_LABELS: False
do_combat: False
ontology: do
scgpt_pp: norm_log1p
description: no control loss
library: RNA-Seq
use_controls: True


In [2]:
import importlib
import scanpy as sc

importlib.reload(ut)
importlib.reload(vz)

split_idx = 0

# load all adata
adata_test =  sc.read(
    os.path.join(run_dir, f"adata_test_{split_idx+1}.h5ad"), backed="r"
)
adata_test.obs.reset_index(drop=True, inplace=True)  
adata_valid =  sc.read(
    os.path.join(run_dir, f"adata_valid_{split_idx+1}.h5ad"), backed="r"
)
adata_valid.obs.reset_index(drop=True, inplace=True)  
adata_train =  sc.read(
    os.path.join(run_dir, f"adata_train_{split_idx+1}.h5ad"), backed="r"
)
adata_train.obs.reset_index(drop=True, inplace=True)  

In [3]:
# load scGPT embeddings
print(f"Using {embedding_type} embeddings")
if embedding_type == "ft":
    embeddings_test = vz.merge_embeddings(all_outputs_test[split_idx])
    embeddings_valid = vz.merge_embeddings(all_outputs_valid[split_idx])
    embeddings_train = vz.merge_embeddings(all_outputs_train[split_idx])
elif embedding_type == "pt":
    pt_adata_test = sc.read(os.path.join("/aloy/scratch/ddalton/projects/scGPT_playground/outputs/",run_name,"pt_adata_test_1.h5ad"))
    pt_adata_valid = sc.read(os.path.join("/aloy/scratch/ddalton/projects/scGPT_playground/outputs/",run_name,"pt_adata_valid_1.h5ad"))
    pt_adata_train = sc.read(os.path.join("/aloy/scratch/ddalton/projects/scGPT_playground/outputs/",run_name,"pt_adata_train_1.h5ad"))
    embeddings_test = pt_adata_test.obsm["X_scGPT"]
    embeddings_valid = pt_adata_valid.obsm["X_scGPT"]
    embeddings_train = pt_adata_train.obsm["X_scGPT"]

    # embeddings_test = np.load(os.path.join("/aloy/scratch/ddalton/projects/scGPT_playground/outputs/",run_name, "test-pt_embeddings.npy"))
    # embeddings_valid = np.load(os.path.join("/aloy/scratch/ddalton/projects/scGPT_playground/outputs/",run_name, "valid-pt_embeddings.npy"))
    # embeddings_train = np.load(os.path.join("/aloy/scratch/ddalton/projects/scGPT_playground/outputs/",run_name, "train-pt_embeddings.npy"))
elif embedding_type == "raw":
    embeddings_test = np.array(adata_test.X)
    embeddings_valid = np.array(adata_valid.X)
    embeddings_train = np.array(adata_train.X)

    # replace nans with 0
    embeddings_test = np.nan_to_num(embeddings_test, nan=0.0)
    embeddings_valid = np.nan_to_num(embeddings_valid, nan=0.0)
    embeddings_train = np.nan_to_num(embeddings_train, nan=0.0)

Using ft embeddings


In [4]:
"""0.2 Load Disease Ontology Pairs
Load pairs of related and unrelated diseases from the Disease Ontology.
Structure:
    1. Imports, Variables, Functions
    2. Load Disease Pairs

"""
# 1. Imports, Variables, Functions
# imports


# variables

# functions


# 2. Load Disease Pairs

# Get disease IDs
d_ids = list(adata_train.obs["doid_id"].unique())

# load Disease Ontology Lin Similarity amongst pairs!
do_df_ic = ut.load_df_do_pairs()
print(f"Loaded universe of disease pairs: {do_df_ic.shape[0]}") 

# Load IC based similarity 
do_graph = ut.load_do_graph()
doid_to_term = {
    node: data["name"] for node, data in do_graph.nodes(data=True) if "name" in data
}


# get top 1% & 5% of values
ic_thr = np.quantile(do_df_ic["lin"].values, 0.95)
print(f"IC Threshold: {ic_thr:.3f}\tNº of pairss {len(do_df_ic.query('lin>@ic_thr'))}")

ic_thr_99 = np.quantile(do_df_ic["lin"].values, 0.99)
print(f"IC Threshold 99%: {ic_thr_99:.3f}\tNº of pairs {len(do_df_ic.query('lin>@ic_thr_99'))}")


# Filter pairs
do_df_ic_query = do_df_ic.query("do1 in @d_ids and do2 in @d_ids")
print("Nº of pairs in dataset:", do_df_ic_query.shape)

do_df_ic_query = do_df_ic.query(f"do1 in @d_ids and do2 in @d_ids and lin > {ic_thr}")
print("Nº of significant pairs in dataset (top 5% IC):", do_df_ic_query.shape)

do_df_ic_query_99 = do_df_ic.query(f"do1 in @d_ids and do2 in @d_ids and lin > {ic_thr_99}")
print("Nº of significant pairs in dataset (top 1% IC):", do_df_ic_query_99.shape)

do_df_negative_ic = do_df_ic.query(f"do1 in @d_ids and do2 in @d_ids and lin <= {ic_thr}")
print("Nº of negative pairs in dataset (outside 5%):", do_df_negative_ic.shape)

# map doid to related diseases
rel_map = {}
for d in d_ids:
    rel_map[d] = set(do_df_ic_query.loc[do_df_ic_query["do1"] == d, "do2"]) | \
                 set(do_df_ic_query.loc[do_df_ic_query["do2"] == d, "do1"])

Loaded universe of disease pairs: 70442515
IC Threshold: 0.396	Nº of pairss 3518554
IC Threshold 99%: 0.543	Nº of pairs 704215
Nº of pairs in dataset: (7503, 8)
Nº of significant pairs in dataset (top 5% IC): (917, 8)
Nº of significant pairs in dataset (top 1% IC): (354, 8)
Nº of negative pairs in dataset (outside 5%): (6586, 8)


# LR: Multilabel 

In [5]:
# imports
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve
from sklearn.metrics import precision_score, recall_score, f1_score

# variables
train_control = True
train_related = False

# functions
def train_disease_model(d:str)->Dict:
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear")

    clf.fit(
        embeddings_train,
        (adata_train.obs["doid_id"] == d).astype(int).values
    )
    
    # train performance
    y_train_prob = clf.predict_proba(embeddings_train)[:, 1]
    y_train_true = (adata_train.obs["doid_id"] == d).astype(int).values

    auc_roc_train = roc_auc_score(y_train_true, y_train_prob)
    ap_train = average_precision_score(y_train_true, y_train_prob)
    pr_train = precision_score(y_train_true, y_train_prob>0.5)
    rc_train = recall_score(y_train_true, y_train_prob>0.5)
    f1_train = f1_score(y_train_true, y_train_prob>0.5)

    # validation performance
    y_valid_prob = clf.predict_proba(embeddings_valid)[:, 1]
    y_valid_true = (adata_valid.obs["doid_id"] == d).astype(int).values

    auc_roc_valid = roc_auc_score(y_valid_true, y_valid_prob)
    ap_valid = average_precision_score(y_valid_true, y_valid_prob)
    pr_valid = precision_score(y_valid_true, y_valid_prob>0.5)
    rc_valid = recall_score(y_valid_true, y_valid_prob>0.5)
    f1_valid = f1_score(y_valid_true, y_valid_prob>0.5)

    # test performance
    y_test_prob = clf.predict_proba(embeddings_test)[:, 1]
    y_test_true = (adata_test.obs["doid_id"] == d).astype(int).values
    auc_roc_test = roc_auc_score(y_test_true, y_test_prob)
    ap_test = average_precision_score(y_test_true, y_test_prob)
    pr_test = precision_score(y_test_true, y_test_prob>0.5)
    rc_test = recall_score(y_test_true, y_test_prob>0.5)
    f1_test = f1_score(y_test_true, y_test_prob>0.5)

    result = {"disease": d,
              "nº_cases_train": sum(y_train_true),
              "nº_cases_valid": sum(y_valid_true),
              "nº_cases_test": sum(y_test_true),
              "AUROC_train": auc_roc_train,
              "AP_train": ap_train,
              "Precision_train": pr_train,
              "Recall_train": rc_train,
              "F1_train": f1_train,
              "AUROC_valid": auc_roc_valid,
              "AP_valid": ap_valid,
              "Precision_valid": pr_valid,
              "Recall_valid": rc_valid,
              "F1_valid": f1_valid,
              "AUROC_test": auc_roc_test,
              "AP_test": ap_test,
              "Precision_test": pr_test,
              "Recall_test": rc_test,
              "F1_test": f1_test,
              }
    return result

from tqdm.contrib.concurrent import process_map
results = process_map(train_disease_model, adata_train.obs["doid_id"].unique()[:10], max_workers=8)

  0%|          | 0/10 [00:00<?, ?it/s]/home/ddalton/Data/old_home/miniconda3/envs/scgpt_2/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ddalton/Data/old_home/miniconda3/envs/scgpt_2/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
100%|██████████| 10/10 [00:21<00:00,  2.15s/it]


In [8]:
pd.DataFrame(results)

,disease,nº_cases_train,nº_cases_valid,nº_cases_test,AUROC_train,AP_train,Precision_train,Recall_train,F1_train,AUROC_valid,AP_valid,Precision_valid,Recall_valid,F1_valid,AUROC_test,AP_test,Precision_test,Recall_test,F1_test
0,Control,6482,720,1637,0.853479,0.749776,0.605573,0.824745,0.698367,0.807828,0.684240,0.563971,0.765278,0.649381,0.780383,0.553655,0.507720,0.682957,0.582443
1,DOID:11714,55,6,26,0.999840,0.906886,0.561224,1.000000,0.718954,0.998767,0.783333,0.500000,0.833333,0.625000,0.866232,0.019286,0.000000,0.000000,0.000000
2,DOID:10286,70,8,14,1.000000,1.000000,0.933333,1.000000,0.965517,1.000000,1.000000,0.888889,1.000000,0.941176,0.993832,0.212501,0.228571,0.571429,0.326531
3,DOID:332,223,25,4,0.999529,0.947594,0.677812,1.000000,0.807971,0.979233,0.812774,0.578947,0.880000,0.698413,0.414092,0.000795,0.000000,0.000000,0.000000
4,DOID:2526,449,50,18,0.999935,0.996983,0.941300,1.000000,0.969762,1.000000,1.000000,1.000000,1.000000,1.000000,0.420480,0.003035,0.000000,0.000000,0.000000
5,DOID:8577,1170,130,40,0.996288,0.944403,0.700180,1.000000,0.823654,0.994061,0.910306,0.656410,0.984615,0.787692,0.934543,0.164312,0.209677,0.325000,0.254902
6,DOID:2841,734,82,16,0.993358,0.809046,0.620981,1.000000,0.766180,0.988748,0.737202,0.645161,0.975610,0.776699,0.665830,0.004508,0.000000,0.000000,0.000000
7,DOID:4947,8,1,36,1.000000,1.000000,0.800000,1.000000,0.888889,1.000000,1.000000,1.000000,1.000000,1.000000,0.878235,0.037093,0.000000,0.000000,0.000000
8,DOID:3910,479,53,8,0.999996,0.999841,0.987629,1.000000,0.993776,0.999914,0.997026,0.898305,1.000000,0.946429,0.637285,0.002247,0.000000,0.000000,0.000000
9,DOID:3068,183,20,6,1.000000,1.000000,0.989189,1.000000,0.994565,0.988133,0.952008,1.000000,0.950000,0.974359,0.999905,0.915079,0.352941,1.000000,0.521739


# Multilabel Classification

In [5]:
# imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import roc_auc_score, average_precision_score

# variables



# functions
class DiseaseNN(nn.Module):
    def __init__(self,input_dim:int, output_dim:int,hid_dim:Tuple=(256, 128), dropout:float=0.1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hid_dim[0])
        self.fc2 = nn.Linear(hid_dim[0], hid_dim[1])
        self.fc3 = nn.Linear(hid_dim[-1], output_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        logits = self.fc3(x)
        return logits

def hit_at_k_from_proba(y_true_int, proba, k=5):
    k = min(k, proba.shape[1])
    topk = np.argsort(-proba, axis=1)[:, :k]
    hits = [int(t in row) for t, row in zip(y_true_int, topk)]
    return float(np.mean(hits)) if len(hits) > 0 else float("nan")

# Filter control samples
remove_controls = False
if remove_controls:
    df_train = adata_train.obs[adata_train.obs["disease"] != "Control"]
    df_valid = adata_valid.obs[adata_valid.obs["disease"] != "Control"]
    df_test  = adata_test.obs [adata_test .obs["disease"] != "Control"]

    e_train = embeddings_train[adata_train.obs["disease"] != "Control"]
    e_valid = embeddings_valid[adata_valid.obs["disease"] != "Control"]
    e_test  = embeddings_test [adata_test .obs["disease"] != "Control"]
else:
    df_train, df_valid, df_test = adata_train.obs, adata_valid.obs, adata_test.obs
    e_train, e_valid, e_test = embeddings_train, embeddings_valid, embeddings_test



In [6]:
# Encode labels
label_map = {k:v for v, k in enumerate(df_train["doid_id"].unique())}
rev_map = {v:k for k,v in label_map.items()}
n_classes = len(label_map)

y_train = np.array([label_map[x] for x in df_train["doid_id"].values])
y_valid = np.array([label_map[x] for x in df_valid["doid_id"].values])
y_test = np.array([label_map[x] for x in df_test["doid_id"].values])

# Convert to tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train = torch.tensor(e_train, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=device)
X_valid = torch.tensor(e_valid, dtype=torch.float32, device=device)
y_valid_tensor = torch.tensor(y_valid, dtype=torch.long, device=device)

# Train
model = DiseaseNN(input_dim=X_train.shape[1], output_dim=n_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 500
patience = 10           # stop if no improvement after these many epochs
best_val_loss = np.inf
patience_counter = 0

for epoch in range(epochs):

    # Train
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss = criterion(logits, y_train_tensor)
    loss.backward()
    optimizer.step()

    # Validate
    model.eval()
    with torch.no_grad():
        val_logits = model(X_valid)
        val_loss = criterion(val_logits, y_valid_tensor).item()

    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict()  # keep best model
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}")
            break

# Load best model weights
model.load_state_dict(best_model_state)

# Evaluate on validation data
with torch.no_grad():
    val_logits = model(X_valid)
    val_proba = torch.softmax(val_logits, dim=1).cpu().numpy()

for k in (1, 5, 10):
    print(f"Hit@{k}: {hit_at_k_from_proba(y_valid, val_proba, k=k):.3f}")


# Evaluate on test data
test_labels = df_test["doid_id"].values
seen_mask = np.isin(test_labels, list(label_map.keys()))
Xv = e_test[seen_mask]
yv_str = test_labels[seen_mask]

print(f"Evaluating on {len(y_test)} test samples across {n_classes} seen classes")

Xv_tensor = torch.tensor(Xv, dtype=torch.float32, device=device)
model.eval()
with torch.no_grad():
    logits = model(Xv_tensor)
    y_proba = torch.softmax(logits, dim=1).cpu().numpy()

# Metrics
y_test_bin = np.zeros((len(y_test), n_classes))
y_test_bin[list(range(len(y_test))), y_test] = 1

auroc_macro = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
auprc_macro = average_precision_score(y_test_bin, y_proba, average="macro")

print(f"Macro AUROC: {auroc_macro:.3f}")
print(f"Macro AUPRC: {auprc_macro:.3f}")

# Per-class metrics
auroc_per_class, auprc_per_class = {}, {}
for c, name in enumerate(label_map.values()):
    y_true_c = (y_test == c).astype(int)
    if y_true_c.sum() == 0:
        continue
    auroc_per_class[name] = roc_auc_score(y_true_c, y_proba[:, c])
    auprc_per_class[name] = average_precision_score(y_true_c, y_proba[:, c])

# Hit@k calculation
for k in (1, 5, 10):
    print(f"Hit@{k}: {hit_at_k_from_proba(y_test, y_proba, k=k):.3f}")


Epoch [1/500] | Train Loss: 4.7908 | Val Loss: 4.6095
Epoch [2/500] | Train Loss: 4.6093 | Val Loss: 4.4311
Epoch [3/500] | Train Loss: 4.4306 | Val Loss: 4.2270
Epoch [4/500] | Train Loss: 4.2250 | Val Loss: 3.9839
Epoch [5/500] | Train Loss: 3.9802 | Val Loss: 3.6960
Epoch [6/500] | Train Loss: 3.6943 | Val Loss: 3.3774
Epoch [7/500] | Train Loss: 3.3759 | Val Loss: 3.0779
Epoch [8/500] | Train Loss: 3.0767 | Val Loss: 2.8886
Epoch [9/500] | Train Loss: 2.8872 | Val Loss: 2.8159
Epoch [10/500] | Train Loss: 2.8111 | Val Loss: 2.7286
Epoch [11/500] | Train Loss: 2.7281 | Val Loss: 2.5901
Epoch [12/500] | Train Loss: 2.5862 | Val Loss: 2.4478
Epoch [13/500] | Train Loss: 2.4436 | Val Loss: 2.3336
Epoch [14/500] | Train Loss: 2.3363 | Val Loss: 2.2506
Epoch [15/500] | Train Loss: 2.2543 | Val Loss: 2.1879
Epoch [16/500] | Train Loss: 2.1913 | Val Loss: 2.1282
Epoch [17/500] | Train Loss: 2.1323 | Val Loss: 2.0590
Epoch [18/500] | Train Loss: 2.0663 | Val Loss: 1.9834
Epoch [19/500] | Tr

In [7]:
# Build array of disease identifiers in output order
dis_order = np.array([rev_map[i] for i in range(n_classes)])

result_test = []
for i in range(len(yv_str)):
    true_doid = yv_str[i]
    pred_scores = y_proba[i]  # length = n_classes

    order = np.argsort(-pred_scores)  # descending order of probabilities
    top1_idx = order[:1]
    top5_idx = order[:5]
    top10_idx = order[:10]

    # Threshold-based masks
    highly_mask = pred_scores >= 0.8
    related_mask = pred_scores >= 0.4

    result_test.append({
        "query_doid": true_doid,
        "top_1": dis_order[top1_idx].tolist(),
        "top_5": dis_order[top5_idx].tolist(),
        "top_10": dis_order[top10_idx].tolist(),
        "highly_related": dis_order[highly_mask].tolist(),
        "related": dis_order[related_mask].tolist(),
    })

df_pred_test = pd.DataFrame(result_test)


In [8]:
# aggregated by samples
format_pct = lambda x: [f"{x_i*100:.2f}" for x_i in x]
perf_results = list()
for k in ["top_1", "top_5", "top_10", "highly_related", "related"]:
    for _df, _condition in zip([df_pred_test], ["No Control"]):

        # group by disease
        _df_counts = get_counts(_df, rel_map, k)
        _hit_same = _df_counts.groupby("query_doid")["hits_same"].mean().mean()
        _hit_rel = _df_counts.groupby("query_doid")["hits_rel"].mean().mean()
        perf_results.append({
            "Key": k,
            "Across": "Diseases",
            "Condition": _condition,
            "Hits Same Disease": f"{_hit_same*100:.1f}",
            "Precision@K": f"{_df_counts.groupby('query_doid')['prec@k'].mean().mean()*100:.1f}",
            "Size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
            "Unique Size": f'{_df_counts["n_total_unique"].mean():.0f}±{_df_counts["n_total_unique"].std():.0f}' if _df_counts["n_total_unique"].std() > 0 else f'{_df_counts["n_total_unique"].mean():.0f}',
            "Hits Related Disease": f"{_hit_rel*100:.1f}",
        })

        # across all samples
        _df_counts = get_counts(_df, rel_map, k)
        _hit_same = _df_counts["hits_same"].mean()
        _hit_rel = _df_counts["hits_rel"].mean()

        perf_results.append({
            "Key": k,
            "Across": "Samples",
            "Condition": _condition,
            "Hits Same Disease": f"{_hit_same*100:.1f}",
            "Precision@K": f"{_df_counts['prec@k'].mean()*100:.1f}",
            "Size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
            "Unique Size": f'{_df_counts["n_total_unique"].mean():.0f}±{_df_counts["n_total_unique"].std():.0f}' if _df_counts["n_total_unique"].std() > 0 else f'{_df_counts["n_total_unique"].mean():.0f}',
            "Hits Related Disease": f"{_hit_rel*100:.1f}",
        })

df_perf_results = pd.DataFrame(perf_results)

df_perf_results

,Key,Across,Condition,Hits Same Disease,Precision@K,Size,Unique Size,Hits Related Disease
0,top_1,Diseases,No Control,10.1,28.6,1,1,18.5
1,top_1,Samples,No Control,27.3,62.8,1,1,35.5
2,top_5,Diseases,No Control,26.0,32.3,5,5,60.1
3,top_5,Samples,No Control,41.2,43.6,5,5,54.2
4,top_10,Diseases,No Control,34.1,30.2,10,10,72.5
5,top_10,Samples,No Control,53.4,38.3,10,10,58.8
6,highly_related,Diseases,No Control,6.7,31.3,1±0,1±0,10.0
7,highly_related,Samples,No Control,17.2,74.4,1±0,1±0,25.6
8,related,Diseases,No Control,10.0,28.6,1±0,1±0,18.4
9,related,Samples,No Control,27.5,64.0,1±0,1±0,35.0


# Regression Prediction

In [9]:
# disease_counts = df_train["doid_id"].value_counts().reindex(dis_order, fill_value=0).values
# disease_counts = np.where(disease_counts == 0, 1, disease_counts)
# weights = 1.0 / disease_counts
# weights = weights / np.mean(weights)
# weights_tensor = torch.tensor(weights, dtype=torch.float32, device=device)

# criterion_bce = nn.BCEWithLogitsLoss(pos_weight=weights_tensor)
# criterion_bce = nn.BCEWithLogitsLoss()

In [10]:
import numpy as np
import torch


from tqdm import tqdm
do_df_ic_filt = do_df_ic.query("do1 in @d_ids and do2 in @d_ids")
doid_to_sim = dict()
for _, r in do_df_ic_filt.iterrows():
    _do1 = r["do1"]
    _do2 = r["do2"]
    _sim = r["lin"]
    doid_to_sim[(_do1, _do2)] = _sim
    doid_to_sim[(_do2, _do1)] = _sim


dis_order = df_train["doid_id"].unique()
print(f"Total dis_ordereases in training set: {len(dis_order)}")

pair_sim = dict()
for d1 in tqdm(dis_order):
    _sim = list()
    for d2 in dis_order:
        if d1 == d2:
            _sim.append(1.0)
        else:
            _sim.append(doid_to_sim[d1, d2] if doid_to_sim[d1, d2]>0.4 else 0.0)
    pair_sim[d1] = _sim

    

# variables
lambda_true = 0.0

# Encode labels
n_classes = len(dis_order)
y_train = np.array([pair_sim[x] for x in df_train["doid_id"].values])
y_valid = np.array([pair_sim[x] for x in df_valid["doid_id"].values])
y_test = np.array([pair_sim[x] for x in df_test["doid_id"].values])

# Convert to tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train = torch.tensor(e_train, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device=device)
X_valid = torch.tensor(e_valid, dtype=torch.float32, device=device)
y_valid_tensor = torch.tensor(y_valid, dtype=torch.float32, device=device)

# Train
model = DiseaseNN(input_dim=X_train.shape[1], output_dim=n_classes).to(device)
criterion = nn.MSELoss()


optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

epochs = 500
patience = 20           # stop if no improvement after these many epochs
best_val_loss = np.inf
patience_counter = 0

for epoch in range(epochs):

    # Train
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss_mse = criterion(logits, y_train_tensor)
    # --- True-label penalty ---
    true_indices = torch.argmax(y_train_tensor, dim=1)               # [B]
    true_logits = logits.gather(1, true_indices.unsqueeze(1)).squeeze()  # [B]
    penalty_loss = ((1.0 - true_logits) ** 2).mean()  # punish low logits at true class

    # Total loss
    loss = loss_mse + lambda_true * penalty_loss
    loss.backward()
    optimizer.step()

    # Validate
    model.eval()
    with torch.no_grad():
        val_logits = model(X_valid)
        val_loss_mse = criterion(val_logits, y_valid_tensor).item()
        val_loss = val_loss_mse 
        scheduler.step(val_loss)

    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict()  # keep best model
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}")
            break

# Load best model weights
model.load_state_dict(best_model_state)

# Evaluate model similarity predictions
model.eval()
with torch.no_grad():
    valid_pred = model(X_valid).cpu().numpy()   # shape: [num_samples, n_classes]

def hit_at_k_from_scores(y_true, y_pred, k=5):
    hits = 0
    for i in range(len(y_true)):
        # Top K predicted disease indices
        topk_pred = np.argsort(y_pred[i])[::-1][:k]
        # Ground truth disease index (the "same disease" — highest similarity = 1)
        true_index = np.argmax(y_true[i])
        if true_index in topk_pred:
            hits += 1
    return hits / len(y_true)

for k in (1, 5, 10):
    hit = hit_at_k_from_scores(y_valid, valid_pred, k)
    print(f"Hit@{k}: {hit:.3f}")


Total dis_ordereases in training set: 124


  0%|          | 0/124 [00:00<?, ?it/s]


KeyError: ('Control', 'DOID:11714')

In [ ]:
# Evaluate on test data
test_labels = df_test["doid_id"].values
seen_mask = np.isin(test_labels, list(label_map.keys()))
Xt = e_test[seen_mask]
yt_str = test_labels[seen_mask]

print(f"Evaluating on {len(Xt)} test samples across {n_classes} seen classes")

# Convert to tensor
Xt_tensor = torch.tensor(Xt, dtype=torch.float32, device=device)

# Predict similarity scores (no softmax!)
model.eval()
with torch.no_grad():
    y_pred = model(Xt_tensor).cpu().numpy()   # shape: [num_samples, n_classes]

# True similarity scores for each disease (same shape)
y_true = np.array([pair_sim[x] for x in yt_str])

# Normalize predictions row-wise (optional but often helpful)
y_pred = y_pred / (y_pred.max(axis=1, keepdims=True) + 1e-8)


def hit_at_k_from_scores(y_true, y_pred, k=5):
    hits = 0
    for i in range(len(y_true)):
        topk_pred = np.argsort(y_pred[i])[::-1][:k]
        true_index = np.argmax(y_true[i])
        if true_index in topk_pred:
            hits += 1
    return hits / len(y_true)


def mean_rank(y_true, y_pred):
    ranks = []
    for i in range(len(y_true)):
        true_index = np.argmax(y_true[i])
        rank = len(y_pred[i]) - np.argsort(y_pred[i]).argsort()[true_index]
        ranks.append(rank)
    return np.mean(ranks)


def mean_reciprocal_rank(y_true, y_pred):
    rr = []
    for i in range(len(y_true)):
        true_index = np.argmax(y_true[i])
        rank = len(y_pred[i]) - np.argsort(y_pred[i]).argsort()[true_index]
        rr.append(1.0 / rank)
    return np.mean(rr)


# Print results
for k in (1, 5, 10):
    print(f"Hit@{k}: {hit_at_k_from_scores(y_true, y_pred, k):.3f}")

print(f"Mean Rank (MR):  {mean_rank(y_true, y_pred):.2f}")
print(f"Mean Reciprocal Rank (MRR): {mean_reciprocal_rank(y_true, y_pred):.3f}")


Evaluating on 3611 test samples across 123 seen classes
Hit@1: 0.074
Hit@5: 0.403
Hit@10: 0.569
Mean Rank (MR):  21.29
Mean Reciprocal Rank (MRR): 0.222


In [ ]:
# Predict similarity probabilities
model.eval()
with torch.no_grad():
    logits = model(Xv_tensor)
    # y_pred = torch.sigmoid(logits).cpu().numpy()  # convert logits → probabilities in [0,1]
    y_pred = model(Xv_tensor).cpu().numpy()



In [ ]:
dis_order = np.array(dis_order)

result_test = list()
for i in range(len(yv_str)):
    true_doid = yv_str[i]
    pred_scores = y_pred[i]
    top10_indices = np.argsort(pred_scores)[::-1][:10]
    top10_doids = [dis_order[idx] for idx in top10_indices]
    result_test.append({
        "query_doid": true_doid,
        "top_1": dis_order[np.argsort(pred_scores)[::-1][:1]],
        "top_5": dis_order[np.argsort(pred_scores)[::-1][:5]],
        "top_10": dis_order[np.argsort(pred_scores)[::-1][:10]],
        "highly_related": dis_order[np.argwhere(pred_scores >= 0.8)].flatten(),
        "related": dis_order[np.argwhere(pred_scores >= 0.4)].flatten(),
    })

df_pred_test = pd.DataFrame(result_test)

In [ ]:


get_counts(df_pred_test, rel_map, key_interest="top_1")

,query_doid,top_1,top_5,top_10,highly_related,related,n_same,n_rel,n_unrel,n_total,n_total_unique,pct_same,pct_rel,pct_unrel,hits_same,hits_rel,prec@k
0,DOID:768,[DOID:768],"[DOID:768, DOID:769, DOID:3068, DOID:3070, DOI...","[DOID:768, DOID:769, DOID:3068, DOID:3070, DOI...",[],[],1,0,0,1,1,100.0,0.0,0.0,1,0,1.0
1,DOID:768,[DOID:768],"[DOID:768, DOID:3068, DOID:769, DOID:3070, DOI...","[DOID:768, DOID:3068, DOID:769, DOID:3070, DOI...",[],[],1,0,0,1,1,100.0,0.0,0.0,1,0,1.0
2,DOID:768,[DOID:768],"[DOID:768, DOID:3068, DOID:769, DOID:3070, DOI...","[DOID:768, DOID:3068, DOID:769, DOID:3070, DOI...",[],[],1,0,0,1,1,100.0,0.0,0.0,1,0,1.0
3,DOID:768,[DOID:768],"[DOID:768, DOID:3068, DOID:769, DOID:3070, DOI...","[DOID:768, DOID:3068, DOID:769, DOID:3070, DOI...",[],[],1,0,0,1,1,100.0,0.0,0.0,1,0,1.0
4,DOID:768,[DOID:768],"[DOID:768, DOID:3068, DOID:769, DOID:3070, DOI...","[DOID:768, DOID:3068, DOID:769, DOID:3070, DOI...",[],[],1,0,0,1,1,100.0,0.0,0.0,1,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3606,DOID:3910,[DOID:3068],"[DOID:3068, DOID:3070, DOID:1749, DOID:2526, D...","[DOID:3068, DOID:3070, DOID:1749, DOID:2526, D...",[],"[DOID:3068, DOID:3070]",0,1,0,1,1,0.0,100.0,0.0,0,1,1.0
3607,DOID:635,[DOID:2377],"[DOID:2377, DOID:8577, DOID:2378, DOID:10608, ...","[DOID:2377, DOID:8577, DOID:2378, DOID:10608, ...","[DOID:8577, DOID:9074, DOID:2377, DOID:8893, D...","[DOID:8577, DOID:7148, DOID:9074, DOID:2377, D...",0,0,1,1,1,0.0,0.0,100.0,0,0,0.0
3608,DOID:635,[DOID:2377],"[DOID:2377, DOID:8577, DOID:10608, DOID:8893, ...","[DOID:2377, DOID:8577, DOID:10608, DOID:8893, ...","[DOID:8577, DOID:2377, DOID:8893, DOID:2378, D...","[DOID:8577, DOID:7148, DOID:9074, DOID:2377, D...",0,0,1,1,1,0.0,0.0,100.0,0,0,0.0
3609,DOID:635,[DOID:2377],"[DOID:2377, DOID:2378, DOID:8577, DOID:9744, D...","[DOID:2377, DOID:2378, DOID:8577, DOID:9744, D...","[DOID:8577, DOID:2377, DOID:8893, DOID:2378, D...","[DOID:8577, DOID:7148, DOID:9074, DOID:2377, D...",0,0,1,1,1,0.0,0.0,100.0,0,0,0.0


In [ ]:
# aggregated by samples
format_pct = lambda x: [f"{x_i*100:.2f}" for x_i in x]
perf_results = list()
for k in ["top_1", "top_5", "top_10", "highly_related", "related"]:
    for _df, _condition in zip([df_pred_test], ["No Control"]):

        # group by disease
        _df_counts = get_counts(_df, rel_map, k)
        _hit_same = _df_counts.groupby("query_doid")["hits_same"].mean().mean()
        _hit_rel = _df_counts.groupby("query_doid")["hits_rel"].mean().mean()
        perf_results.append({
            "Key": k,
            "Across": "Diseases",
            "Condition": _condition,
            "Type": "Same Disease",
            "Hits Same Disease": f"{_hit_same*100:.1f}",
            "Precision@K": f"{_df_counts.groupby('query_doid')['prec@k'].mean().mean()*100:.1f}",
            "Size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
            "Unique Size": f'{_df_counts["n_total_unique"].mean():.0f}±{_df_counts["n_total_unique"].std():.0f}' if _df_counts["n_total_unique"].std() > 0 else f'{_df_counts["n_total_unique"].mean():.0f}',
            "Hits Related Disease": f"{_hit_rel*100:.1f}",
        })

        # across all samples
        _df_counts = get_counts(_df, rel_map, k)
        _hit_same = _df_counts["hits_same"].mean()
        _hit_rel = _df_counts["hits_rel"].mean()

        perf_results.append({
            "Key": k,
            "Across": "Samples",
            "Condition": _condition,
            "Type": "Related Disease",
            "Hits Same Disease": f"{_hit_same*100:.1f}",
            "Precision@K": f"{_df_counts['prec@k'].mean()*100:.1f}",
            "Size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
            "Unique Size": f'{_df_counts["n_total_unique"].mean():.0f}±{_df_counts["n_total_unique"].std():.0f}' if _df_counts["n_total_unique"].std() > 0 else f'{_df_counts["n_total_unique"].mean():.0f}',
            "Hits Related Disease": f"{_hit_rel*100:.1f}",
        })

df_perf_results = pd.DataFrame(perf_results)

df_perf_results

,Key,Across,Condition,Type,Hits Same Disease,Precision@K,Size,Unique Size,Hits Related Disease
0,top_1,Diseases,No Control,Same Disease,10.1,51.1,1,1,40.9
1,top_1,Samples,No Control,Related Disease,7.4,73.7,1,1,66.3
2,top_5,Diseases,No Control,Same Disease,26.7,46.9,5,5,59.7
3,top_5,Samples,No Control,Related Disease,40.3,68.7,5,5,77.8
4,top_10,Diseases,No Control,Same Disease,37.0,44.7,10,10,65.1
5,top_10,Samples,No Control,Related Disease,56.9,68.3,10,10,80.9
6,highly_related,Diseases,No Control,Same Disease,4.9,68.6,1±1,1±1,9.6
7,highly_related,Samples,No Control,Related Disease,4.6,90.1,1±1,1±1,26.7
8,related,Diseases,No Control,Same Disease,27.2,50.2,16±13,16±13,41.3
9,related,Samples,No Control,Related Disease,60.9,79.4,16±13,16±13,69.3


# Multiclass Multilabel

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# 1. Build pairwise similarity bins (0, 1, 2)
do_df_ic_filt = do_df_ic.query("do1 in @d_ids and do2 in @d_ids")
doid_to_sim = dict()
for _, r in do_df_ic_filt.iterrows():
    _do1, _do2, _sim = r["do1"], r["do2"], r["lin"]
    doid_to_sim[(_do1, _do2)] = _sim
    doid_to_sim[(_do2, _do1)] = _sim

dis_order = df_train["doid_id"].unique()
print(f"Total diseases in training set: {len(dis_order)}")

pair_sim = dict()
for d1 in tqdm(dis_order):
    _sim = []
    for d2 in dis_order:
        if d1 == d2:
            _sim.append(2)  # SAME disease
        else:
            s = doid_to_sim.get((d1, d2), 0.0)
            if s > 0.4:
                _sim.append(1)  # moderately similar
            else:
                _sim.append(0)  # unrelated
    pair_sim[d1] = _sim

# 2. Encode labels (now integers 0, 1, 2)
n_classes = 3  # three bins: 0, 1, 2
n_diseases = len(dis_order)

y_train = np.array([pair_sim[x] for x in df_train["doid_id"].values])
y_valid = np.array([pair_sim[x] for x in df_valid["doid_id"].values])
y_test  = np.array([pair_sim[x] for x in df_test["doid_id"].values])

# Convert to torch tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train = torch.tensor(e_train, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=device)  # int labels
X_valid = torch.tensor(e_valid, dtype=torch.float32, device=device)
y_valid_tensor = torch.tensor(y_valid, dtype=torch.long, device=device)

# 3. Define model and loss
# Model: predicts a probability distribution over 3 classes per disease
model = DiseaseNN(input_dim=X_train.shape[1], output_dim=n_diseases * n_classes).to(device)

# Helper: reshape output to [batch_size, n_diseases, n_classes]
def reshape_logits(logits):
    return logits.view(-1, n_diseases, n_classes)

# Flatten across all diseases
all_train_labels = y_train.flatten()

# Count occurrences of each class (0,1,2)
class_counts = np.bincount(all_train_labels, minlength=3)
print("Class counts:", class_counts)

# Inverse-frequency weights
class_weights = 1.0 / (class_counts + 1e-8)
class_weights = class_weights / class_weights.mean()  # normalize for stability
print("Class weights:", class_weights)

# Convert to torch tensor on the correct device
weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.1)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
# criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# 4. Training loop
epochs = 500
patience = 20
best_val_loss = np.inf
patience_counter = 0

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    logits = model(X_train)
    logits = reshape_logits(logits)  # [B, n_diseases, n_classes]

    # CrossEntropyLoss expects [B, C, *] and int targets
    loss = criterion(logits.permute(0, 2, 1), y_train_tensor)
    loss.backward()
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_logits = model(X_valid)
        val_logits = reshape_logits(val_logits)
        val_loss = criterion(val_logits.permute(0, 2, 1), y_valid_tensor).item()
        scheduler.step(val_loss)

    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_model_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}")
            break

model.load_state_dict(best_model_state)

# 5. Evaluate: convert logits → probabilities
model.eval()
with torch.no_grad():
    val_logits = model(X_valid)
    val_logits = reshape_logits(val_logits)
    val_probs = torch.softmax(val_logits, dim=-1).cpu().numpy()  # [B, n_diseases, 3]
    # Take probability of being class "2" (same disease)
    valid_pred = val_probs[:, :, 2]

# 6. Metrics (Hit@K etc.)
def hit_at_k_from_scores(y_true, y_pred, k=5):
    hits = 0
    for i in range(len(y_true)):
        topk_pred = np.argsort(y_pred[i])[::-1][:k]
        true_index = np.argmax(y_true[i])
        if true_index in topk_pred:
            hits += 1
    return hits / len(y_true)

for k in (1, 5, 10):
    hit = hit_at_k_from_scores(y_valid, valid_pred, k)
    print(f"Hit@{k}: {hit:.3f}")


Total diseases in training set: 124


100%|██████████| 124/124 [00:00<00:00, 25074.42it/s]

Class counts: [2042890  208625   18305]
Class weights: [0.0245108  0.24001379 2.7354754 ]
Epoch [1/500] | Train Loss: 1.1198 | Val Loss: 1.0665
Epoch [2/500] | Train Loss: 1.0671 | Val Loss: 1.0199
Epoch [3/500] | Train Loss: 1.0201 | Val Loss: 0.9699
Epoch [4/500] | Train Loss: 0.9700 | Val Loss: 0.9141
Epoch [5/500] | Train Loss: 0.9151 | Val Loss: 0.8549
Epoch [6/500] | Train Loss: 0.8569 | Val Loss: 0.7963
Epoch [7/500] | Train Loss: 0.7997 | Val Loss: 0.7425
Epoch [8/500] | Train Loss: 0.7479 | Val Loss: 0.6971
Epoch [9/500] | Train Loss: 0.7045 | Val Loss: 0.6623
Epoch [10/500] | Train Loss: 0.6716 | Val Loss: 0.6355
Epoch [11/500] | Train Loss: 0.6469 | Val Loss: 0.6114
Epoch [12/500] | Train Loss: 0.6227 | Val Loss: 0.5860
Epoch [13/500] | Train Loss: 0.5978 | Val Loss: 0.5585
Epoch [14/500] | Train Loss: 0.5706 | Val Loss: 0.5300
Epoch [15/500] | Train Loss: 0.5419 | Val Loss: 0.5025
Epoch [16/500] | Train Loss: 0.5146 | Val Loss: 0.4774
Epoch [17/500] | Train Loss: 0.4885 | V


/home/ddalton/Data/old_home/miniconda3/envs/scgpt_2/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch [51/500] | Train Loss: 0.2061 | Val Loss: 0.2036
Epoch [52/500] | Train Loss: 0.2032 | Val Loss: 0.2012
Epoch [53/500] | Train Loss: 0.2002 | Val Loss: 0.1990
Epoch [54/500] | Train Loss: 0.1980 | Val Loss: 0.1968
Epoch [55/500] | Train Loss: 0.1959 | Val Loss: 0.1948
Epoch [56/500] | Train Loss: 0.1927 | Val Loss: 0.1929
Epoch [57/500] | Train Loss: 0.1910 | Val Loss: 0.1910
Epoch [58/500] | Train Loss: 0.1886 | Val Loss: 0.1893
Epoch [59/500] | Train Loss: 0.1861 | Val Loss: 0.1875
Epoch [60/500] | Train Loss: 0.1849 | Val Loss: 0.1859
Epoch [61/500] | Train Loss: 0.1828 | Val Loss: 0.1844
Epoch [62/500] | Train Loss: 0.1811 | Val Loss: 0.1829
Epoch [63/500] | Train Loss: 0.1782 | Val Loss: 0.1815
Epoch [64/500] | Train Loss: 0.1775 | Val Loss: 0.1801
Epoch [65/500] | Train Loss: 0.1763 | Val Loss: 0.1788
Epoch [66/500] | Train Loss: 0.1739 | Val Loss: 0.1774
Epoch [67/500] | Train Loss: 0.1734 | Val Loss: 0.1762
Epoch [68/500] | Train Loss: 0.1722 | Val Loss: 0.1749
Epoch [69/

In [ ]:
model.eval()
with torch.no_grad():
    val_logits = model(X_valid)
    val_logits = reshape_logits(val_logits)
    val_probs = torch.softmax(val_logits, dim=-1).cpu().numpy()  # [B, n_diseases, 3]
    # Take probability of being class "2" (same disease)
    valid_pred = val_probs[:, :, 2]

In [ ]:
import numpy as np
import pandas as pd
import torch

# --- Evaluate on test data ---
# 'dis_order' = array of diseases seen in training (same order used to build targets)
dis_order = np.array(dis_order)

test_labels = df_test["doid_id"].values
seen_mask = np.isin(test_labels, dis_order)   # seen classes only
Xv = e_test[seen_mask]
yv_str = test_labels[seen_mask]

print(f"Evaluating on {len(Xv)} test samples across {len(dis_order)} seen diseases")

# Tensor
Xv_tensor = torch.tensor(Xv, dtype=torch.float32, device=device)

# --- Predict probabilities ---
model.eval()
with torch.no_grad():
    logits = model(Xv_tensor)  # shape [B, n_diseases*3] if you followed the earlier code
    # reshape to [B, n_diseases, 3]
    n_diseases = len(dis_order)
    logits = logits.view(-1, n_diseases, 3)
    probs = torch.softmax(logits, dim=-1)           # [B, n_diseases, 3]
    y_pred = probs[:, :, 2].cpu().numpy()           # P(class=2) = same disease

# --- Build ranked predictions + thresholded lists ---
result_test = []
for i in range(len(yv_str)):
    true_doid = yv_str[i]
    pred_scores = y_pred[i]                         # length = n_diseases

    order = np.argsort(-pred_scores)                # descending
    top1_idx = order[:1]
    top5_idx = order[:5]
    top10_idx = order[:10]

    # thresholds on P(class=2)
    highly_mask = pred_scores >= 0.8
    related_mask = pred_scores >= 0.5

    result_test.append({
        "query_doid": true_doid,
        "top_1": dis_order[top1_idx],
        "top_5": dis_order[top5_idx],
        "top_10": dis_order[top10_idx],
        "highly_related": dis_order[highly_mask],
        "related": dis_order[related_mask],
    })

df_pred_test = pd.DataFrame(result_test)



Evaluating on 5248 test samples across 124 seen diseases


In [ ]:
# aggregated by samples
format_pct = lambda x: [f"{x_i*100:.2f}" for x_i in x]
perf_results = list()

for k in ["top_1", "top_5", "top_10", "highly_related", "related"]:
    for _df, _condition in zip([df_pred_test], ["No Control"]):

        # group by disease
        _df_counts = get_counts(_df, rel_map, k)
        _hit_same = _df_counts.groupby("query_doid")["hits_same"].mean().mean()
        _hit_rel = _df_counts.groupby("query_doid")["hits_rel"].mean().mean()
        perf_results.append({
            "Key": k,
            "Across": "Diseases",
            "Condition": _condition,
            "Hits Same Disease": f"{_hit_same*100:.1f}",
            "Precision@K": f"{_df_counts.groupby('query_doid')['prec@k'].mean().mean()*100:.1f}",
            "Size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
            "Unique Size": f'{_df_counts["n_total_unique"].mean():.0f}±{_df_counts["n_total_unique"].std():.0f}' if _df_counts["n_total_unique"].std() > 0 else f'{_df_counts["n_total_unique"].mean():.0f}',
            "Hits Related Disease": f"{_hit_rel*100:.1f}",
        })

        # across all samples
        _df_counts = get_counts(_df, rel_map, k)
        _hit_same = _df_counts["hits_same"].mean()
        _hit_rel = _df_counts["hits_rel"].mean()

        perf_results.append({
            "Key": k,
            "Across": "Samples",
            "Condition": _condition,
            "Hits Same Disease": f"{_hit_same*100:.1f}",
            "Precision@K": f"{_df_counts['prec@k'].mean()*100:.1f}",
            "Size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
            "Unique Size": f'{_df_counts["n_total_unique"].mean():.0f}±{_df_counts["n_total_unique"].std():.0f}' if _df_counts["n_total_unique"].std() > 0 else f'{_df_counts["n_total_unique"].mean():.0f}',
            "Hits Related Disease": f"{_hit_rel*100:.1f}",
        })

df_perf_results = pd.DataFrame(perf_results)

df_perf_results

,Key,Across,Condition,Hits Same Disease,Precision@K,Size,Unique Size,Hits Related Disease
0,top_1,Diseases,No Control,7.1,21.9,1,1,14.8
1,top_1,Samples,No Control,29.1,63.2,1,1,34.1
2,top_5,Diseases,No Control,25.3,29.5,5,5,57.7
3,top_5,Samples,No Control,41.7,39.8,5,5,52.8
4,top_10,Diseases,No Control,35.3,28.6,10,10,70.2
5,top_10,Samples,No Control,49.2,35.7,10,10,57.0
6,highly_related,Diseases,No Control,12.2,24.9,1±1,1±1,25.3
7,highly_related,Samples,No Control,33.1,60.6,1±1,1±1,37.3
8,related,Diseases,No Control,16.4,27.8,2±1,2±1,35.7
9,related,Samples,No Control,36.5,56.1,2±1,2±1,42.4


## Logistic Regression Multilabel Classification

In [9]:
# Logistic Regression with OneVsRestClassifier for probability-based metrics (AUROC, AUPRC, Hit@k)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.multiclass import OneVsRestClassifier

# filter controls out
remove_controls = True

if remove_controls:
    df_train = adata_train.obs[adata_train.obs["disease"] != "Control"]
    df_valid = adata_valid.obs[adata_valid.obs["disease"] != "Control"]
    df_test = adata_test.obs[adata_test.obs["disease"] != "Control"]

    e_test = embeddings_test[adata_test.obs["disease"] != "Control"]
    e_valid = embeddings_valid[adata_valid.obs["disease"] != "Control"]
    e_train = embeddings_train[adata_train.obs["disease"] != "Control"]
else:
    df_train = adata_train.obs
    df_valid = adata_valid.obs
    df_test = adata_test.obs

    e_test = embeddings_test
    e_valid = embeddings_valid
    e_train = embeddings_train

# 1) Encode labels using TRAIN classes only (no LabelEncoder)
label_map = {k: v for v, k in enumerate(df_train["doid_id"].unique())}
rev_map = {v: k for k, v in label_map.items()}
n_classes = len(label_map)

y_train = np.array([label_map[x] for x in df_train["doid_id"].values])

# train classifier
clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced"), multilabel_=True, n_jobs=8)
clf.fit(e_train, y_train)

# 2) Evaluation restricted to doid_ids seen in TRAIN
test_labels = df_test["doid_id"].values
seen_mask = np.isin(test_labels, list(label_map.keys()))
Xv = e_test[seen_mask]
yv_str = test_labels[seen_mask]
y_test = np.array([label_map[x] for x in yv_str])  # integer labels (seen only)

print(f"Evaluating on {len(y_test)} test cells from {n_classes} seen classes")
print(f"Nº of embeddings {Xv.shape} from original {e_test.shape}")

# 3) Probabilities (no hard predict)
y_proba = clf.predict_proba(Xv)  # shape (n_samples, n_train_classes)

# 4) Probability-based metrics
#    (a) Macro AUROC (OvR) and Macro AUPRC over seen classes
y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))
auroc_macro = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
auprc_macro = average_precision_score(y_test_bin, y_proba, average="macro")

print(f"Macro AUROC (seen classes): {auroc_macro:.3f}")
print(f"Macro AUPRC (seen classes): {auprc_macro:.3f}")

#    (b) Per-class AUROC/AUPRC
auroc_per_class = {}
auprc_per_class = {}
for c, name in rev_map.items():
    y_true_c = (y_test == c).astype(int)
    if y_true_c.sum() == 0:
        continue
    auroc_per_class[name] = roc_auc_score(y_true_c, y_proba[:, c])
    auprc_per_class[name] = average_precision_score(y_true_c, y_proba[:, c])

# 5) Optional: Top-k from probabilities
def hit_at_k_from_proba(y_true_int, proba, k=5):
    topk = np.argsort(-proba, axis=1)[:, :k]
    hits = [int(t in row) for t, row in zip(y_true_int, topk)]
    return np.mean(hits)

for k in (1, 5, 10):
    print(f"Hit@{k} (seen classes): {hit_at_k_from_proba(y_test, y_proba, k=k):.3f}")


TypeError: OneVsRestClassifier.__init__() got an unexpected keyword argument 'multilabel_'

In [ ]:
np.sum(y_proba[0])

1.0000000000000002

In [ ]:
import pandas as pd
import numpy as np

# Get top-1 predicted label index for each sample
top1_idx = np.argmax(y_proba, axis=1)

# Convert integer labels back to disease strings using rev_map
true_disease = [rev_map[i] for i in y_test]
pred_top1 = [rev_map[i] for i in top1_idx]

# Create a DataFrame
df_predictions = pd.DataFrame({
    "query_doid": true_disease,
    "predicted_top1": [[p] for p in pred_top1]
})

# Optional: show first few rows
print(df_predictions.head())


  query_doid predicted_top1
0   DOID:768     [DOID:768]
1   DOID:768     [DOID:768]
2   DOID:768     [DOID:768]
3   DOID:768     [DOID:768]
4   DOID:768     [DOID:768]


In [ ]:
k = "predicted_top1"

# group by disease
_df_counts = get_counts(df_predictions, rel_map, k)
_hit_same = _df_counts.groupby("query_doid")["hits_same"].mean().mean()
_hit_rel = _df_counts.groupby("query_doid")["hits_rel"].mean().mean()

perf_results = list()
perf_results.append({
    "Key": k,
    "Across": "Diseases",
    "Condition": _condition,
    "Model": "MultiLabel LR",
    "Type": "Same Disease",
    "Hits Same Disease": f"{_hit_same*100:.1f}",
    "Precision@K": f"{_df_counts.groupby('query_doid')['prec@k'].mean().mean()*100:.1f}",
    "Set size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
    "Hits Related Disease": f"{_hit_rel*100:.1f}",
})

# across all samples
_df_counts = get_counts(df_predictions, rel_map, k)
_hit_same = _df_counts["hits_same"].mean()
_hit_rel = _df_counts["hits_rel"].mean()

perf_results.append({
    "Key": k,
    "Across": "Samples",
    "Condition": _condition,
    "Model": "MultiLabel LR",
    "Type": "Related Disease",
    "Hits Same Disease": f"{_hit_same*100:.1f}",
    "Precision@K": f"{_df_counts['prec@k'].mean()*100:.1f}",
    "Set size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
    "Hits Related Disease": f"{_hit_rel*100:.1f}",
})

df_perf_results = pd.DataFrame(perf_results)


In [ ]:
df_perf_results

,Key,Across,Condition,Model,Type,Hits Same Disease,Precision@K,Set size,Hits Related Disease
0,predicted_top1,Diseases,No Control,MultiLabel LR,Same Disease,12.8,46.8,1,34.0
1,predicted_top1,Samples,No Control,MultiLabel LR,Related Disease,8.3,70.0,1,61.7


## Multilabel LR

In [ ]:
# Logistic Regression with OneVsRestClassifier for multilabel probability-based metrics (AUROC, AUPRC, Hit@k)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.multiclass import OneVsRestClassifier

# --------------------------
# 0) Filter out control samples
# --------------------------
remove_controls = True

if remove_controls:
    df_train = adata_train.obs[adata_train.obs["disease"] != "Control"]
    df_valid = adata_valid.obs[adata_valid.obs["disease"] != "Control"]
    df_test = adata_test.obs[adata_test.obs["disease"] != "Control"]

    e_test = embeddings_test[adata_test.obs["disease"] != "Control"]
    e_valid = embeddings_valid[adata_valid.obs["disease"] != "Control"]
    e_train = embeddings_train[adata_train.obs["disease"] != "Control"]
else:
    df_train = adata_train.obs
    df_valid = adata_valid.obs
    df_test = adata_test.obs

    e_test = embeddings_test
    e_valid = embeddings_valid
    e_train = embeddings_train

# 1) Encode labels using TRAIN classes only (no LabelEncoder)
dis_order = df_train["doid_id"].unique()
label_map = {k: v for v, k in enumerate(dis_order)}
rev_map = {v: k for k, v in label_map.items()}
n_classes = len(label_map)

y_train = np.array([label_map[x] for x in df_train["doid_id"].values])
y_valid = np.array([label_map[x] for x in df_valid["doid_id"].values])
y_test = np.array([label_map[x] for x in df_test["doid_id"].values])



In [ ]:
df_train["doid_id"]

1        DOID:11714
4        DOID:10286
5          DOID:332
6         DOID:2526
7         DOID:8577
            ...    
18298     DOID:9074
18300      DOID:684
18301     DOID:3910
18302    DOID:11714
18304     DOID:1520
Name: doid_id, Length: 11823, dtype: category
Categories (124, object): ['Control', 'DOID:235', 'DOID:289', 'DOID:332', ..., 'DOID:0060161', 'DOID:0060488', 'DOID:0080199', 'DOID:0081087']

In [ ]:
label_map

{'0': 0,
 '1': 1,
 '2': 2,
 '3': 3,
 '4': 4,
 '5': 5,
 '6': 6,
 '7': 7,
 '8': 8,
 '9': 9,
 ':': 10,
 'D': 11,
 'I': 12,
 'O': 13}

In [ ]:
df_train["doid_id"]

1        DOID:11714
4        DOID:10286
5          DOID:332
6         DOID:2526
7         DOID:8577
            ...    
18298     DOID:9074
18300      DOID:684
18301     DOID:3910
18302    DOID:11714
18304     DOID:1520
Name: doid_id, Length: 11823, dtype: category
Categories (124, object): ['Control', 'DOID:235', 'DOID:289', 'DOID:332', ..., 'DOID:0060161', 'DOID:0060488', 'DOID:0080199', 'DOID:0081087']

In [ ]:
np.sum(y_proba[0])

1.0

In [ ]:
k = "predicted_top1"

# group by disease
_df_counts = get_counts(df_predictions, rel_map, k)
_hit_same = _df_counts.groupby("query_doid")["hits_same"].mean().mean()
_hit_rel = _df_counts.groupby("query_doid")["hits_rel"].mean().mean()

perf_results = list()
perf_results.append({
    "Key": k,
    "Across": "Diseases",
    "Condition": _condition,
    "Model": "MultiLabel LR",
    "Type": "Same Disease",
    "Hits Same Disease": f"{_hit_same*100:.1f}",
    "Precision@K": f"{_df_counts.groupby('query_doid')['prec@k'].mean().mean()*100:.1f}",
    "Set size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
    "Hits Related Disease": f"{_hit_rel*100:.1f}",
})

# across all samples
_df_counts = get_counts(df_predictions, rel_map, k)
_hit_same = _df_counts["hits_same"].mean()
_hit_rel = _df_counts["hits_rel"].mean()

perf_results.append({
    "Key": k,
    "Across": "Samples",
    "Condition": _condition,
    "Model": "MultiLabel LR",
    "Type": "Related Disease",
    "Hits Same Disease": f"{_hit_same*100:.1f}",
    "Precision@K": f"{_df_counts['prec@k'].mean()*100:.1f}",
    "Set size": f'{_df_counts["n_total"].mean():.0f}±{_df_counts["n_total"].std():.0f}' if _df_counts["n_total"].std() > 0 else f'{_df_counts["n_total"].mean():.0f}',
    "Hits Related Disease": f"{_hit_rel*100:.1f}",
})

df_perf_results = pd.DataFrame(perf_results)
df_perf_results

,Key,Across,Condition,Model,Type,Hits Same Disease,Precision@K,Set size,Hits Related Disease
0,predicted_top1,Diseases,No Control,MultiLabel LR,Same Disease,13.1,46.8,1,33.7
1,predicted_top1,Samples,No Control,MultiLabel LR,Related Disease,8.8,70.4,1,61.6
